# Experiment 1 — Relaxed Modularity Scoring

Compute a modularity score for all universal circuits at four values of the relaxation parameter p = {0, 0.05, 0.10, 0.25}.

**Original modularity** (p=0): Is circuit X's mean Jaccard similarity to ALL other circuits in the bottom 5% of the population at this layer?

**Relaxed modularity** (p>0): After dropping the top p fraction of most-similar neighbors, is circuit X's trimmed mean Jaccard in the bottom 5% of the population's trimmed means at this layer?

At p=0 this reproduces notebook 4's results exactly. At p=0.25 we allow up to ~26 "friends" before testing.

**Method:** Population percentile (no permutation test). For each object at each layer, compute where it ranks among all objects. Deterministic, fast, and consistent with notebook 4.

**Output:** A table of shape N_objects × 4, plus per-layer detail CSVs.

In [ ]:
# Cell 1 – Configuration
import numpy as np

SIGNIFICANCE = 0.05
P_VALUES = [0.0, 0.05, 0.10, 0.25]
N_LAYERS = 8

print(f"Significance: {SIGNIFICANCE}")
print(f"P values: {P_VALUES}")

In [ ]:
# Cell 2 – Imports
import subprocess, sys, os, shutil
for pkg in ["h5py", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import h5py
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("Imports OK")

In [ ]:
# Cell 3 – Mount Google Drive / detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/DATA/CSP-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/CSP-Atlas")

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Cell 4 – Load HDF5, extract all universal masks
UNIVERSAL_FILE = "universal_106x50.h5"
H5_PATH = DATA_DIR / UNIVERSAL_FILE
assert H5_PATH.exists(), f"File not found: {H5_PATH}"
print(f"Loading: {H5_PATH}")

atlas = h5py.File(H5_PATH, "r")
um = atlas["universal_masks"]

masks = {}  # {name: {layer: np.array(bool, 2048)}}
obj_types = {}  # {name: "ast" or "builtin"}

for layer_id in range(N_LAYERS):
    layer_key = f"layer_{layer_id}"
    if layer_key not in um:
        continue
    for ds_name in um[layer_key]:
        # Detect separator: try "__" first, then "_"
        if "__" in ds_name:
            parts = ds_name.split("__", 1)
        else:
            parts = ds_name.split("_", 1)
        obj_type = parts[0]   # "ast" or "builtin"
        full_name = ds_name   # keep full name as unique key

        if full_name not in masks:
            masks[full_name] = {}
            obj_types[full_name] = obj_type

        masks[full_name][layer_id] = np.array(um[layer_key][ds_name], dtype=bool)

all_objects = sorted(masks.keys())
N_OBJECTS = len(all_objects)
print(f"Loaded {N_OBJECTS} universal circuits")
print(f"  AST:     {sum(1 for v in obj_types.values() if v == 'ast')}")
print(f"  Builtin: {sum(1 for v in obj_types.values() if v == 'builtin')}")

# Sanity check: each object should have all 8 layers
for obj in all_objects:
    assert len(masks[obj]) == N_LAYERS, f"{obj} has {len(masks[obj])} layers, expected {N_LAYERS}"

# Close HDF5 — all data is in numpy arrays now
atlas.close()
print("HDF5 closed, all masks in memory.")

In [ ]:
# Cell 5 – Diagnostic: confirm HDF5 key format
# Re-open briefly to inspect
with h5py.File(H5_PATH, "r") as f:
    um_diag = f["universal_masks"]
    layer_key = "layer_0"
    if layer_key in um_diag:
        sample_names = list(um_diag[layer_key].keys())[:10]
        print(f"Sample dataset names in {layer_key}:")
        for name in sample_names:
            shape = um_diag[layer_key][name].shape
            print(f"  {name}: shape={shape}")
    else:
        print(f"Layer key '{layer_key}' not found. Available keys: {list(um_diag.keys())}")

print(f"\nTotal objects loaded: {N_OBJECTS}")
print(f"Sample object names: {all_objects[:5]}")

In [ ]:
# Cell 6 – Precompute mask_matrices and jaccard_matrices

def jaccard_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Jaccard = |A ∩ B| / |A ∪ B|. Returns 0.0 if both empty."""
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 0.0
    return float(intersection / union)

# Stacked mask matrices for fast vectorized operations
mask_matrices = {}
for layer_id in range(N_LAYERS):
    mat = np.zeros((N_OBJECTS, 2048), dtype=bool)
    for i, obj_name in enumerate(all_objects):
        mat[i] = masks[obj_name][layer_id]
    mask_matrices[layer_id] = mat

# Full Jaccard matrices
jaccard_matrices = {}
for layer_id in tqdm(range(N_LAYERS), desc="Jaccard matrices"):
    mat = np.zeros((N_OBJECTS, N_OBJECTS), dtype=np.float32)
    for i in range(N_OBJECTS):
        for j in range(i + 1, N_OBJECTS):
            sim = jaccard_similarity(
                masks[all_objects[i]][layer_id],
                masks[all_objects[j]][layer_id]
            )
            mat[i, j] = sim
            mat[j, i] = sim
        mat[i, i] = 1.0
    jaccard_matrices[layer_id] = mat

print(f"Precomputed {N_LAYERS} Jaccard matrices of shape ({N_OBJECTS}, {N_OBJECTS})")
print(f"Precomputed {N_LAYERS} mask matrices of shape ({N_OBJECTS}, 2048)")

In [ ]:
# Cell 7 – Helper functions

def trimmed_mean_jaccard(jaccard_row: np.ndarray, self_idx: int, p: float) -> float:
    """
    Compute mean Jaccard after dropping top-p fraction of most similar neighbors.
    
    Args:
        jaccard_row: Full row from Jaccard matrix, shape (N_OBJECTS,)
        self_idx: Index of the circuit itself (to exclude self-similarity)
        p: Fraction of nearest neighbors to drop (0 = strict, 0.25 = drop top 25%)
    
    Returns:
        Trimmed mean Jaccard similarity
    """
    others = np.delete(jaccard_row, self_idx)
    n_others = len(others)
    
    k = int(np.floor(p * n_others))
    
    if k > 0:
        sorted_desc = np.sort(others)[::-1]
        trimmed = sorted_desc[k:]
    else:
        trimmed = others
    
    if len(trimmed) == 0:
        return 0.0
    
    return float(trimmed.mean())


print("Helper functions defined.")

In [ ]:
# Cell 8 – Population percentile test

def percentile_test_all_p_values(
    circuit_idx: int,
    layer_id: int,
    p_values: list,
    all_trimmed_means: dict,
) -> dict:
    """
    Test whether circuit_idx has a trimmed-mean Jaccard in the bottom
    SIGNIFICANCE fraction of the population, for all p values at once.
    
    Args:
        circuit_idx: index of the circuit to test
        layer_id: which layer
        p_values: list of relaxation parameters
        all_trimmed_means: {p: np.array of shape (N_OBJECTS,)} precomputed
    
    Returns:
        {p_val: (observed, percentile, is_significant)}
    """
    results = {}
    for p in p_values:
        observed = all_trimmed_means[p][circuit_idx]
        # What fraction of the population has trimmed-mean-J <= this object?
        percentile = float((all_trimmed_means[p] <= observed).sum() / N_OBJECTS)
        is_sig = percentile < SIGNIFICANCE
        results[p] = (observed, percentile, is_sig)
    return results


print("Percentile test function defined.")

In [ ]:
# Cell 9 – Main loop: compute all scores

results = np.zeros((N_OBJECTS, len(P_VALUES)), dtype=int)
detail_records = []

for layer_id in tqdm(range(N_LAYERS), desc="Layers"):
    # Precompute trimmed means for ALL objects at this layer, for each p
    all_trimmed_means = {}
    for p_val in P_VALUES:
        means = np.array([
            trimmed_mean_jaccard(jaccard_matrices[layer_id][i], i, p_val)
            for i in range(N_OBJECTS)
        ], dtype=np.float32)
        all_trimmed_means[p_val] = means
    
    # Test each object
    for obj_idx in range(N_OBJECTS):
        obj_name = all_objects[obj_idx]
        layer_results = percentile_test_all_p_values(
            obj_idx, layer_id, P_VALUES, all_trimmed_means
        )
        
        for p_col, p_val in enumerate(P_VALUES):
            observed, percentile, is_sig = layer_results[p_val]
            
            if is_sig:
                results[obj_idx, p_col] += 1
            
            detail_records.append({
                "object": obj_name,
                "type": obj_types[obj_name],
                "layer": layer_id,
                "p_trim": p_val,
                "observed_trimmed_jaccard": round(observed, 4),
                "percentile": round(percentile, 4),
                "significant": is_sig
            })

print(f"Done. Results shape: {results.shape}")

In [ ]:
# Cell 10 – Build results table and save CSVs
results_df = pd.DataFrame(
    results,
    index=all_objects,
    columns=[f"p={p}" for p in P_VALUES]
)
results_df.insert(0, "type", [obj_types[obj] for obj in all_objects])

# Sort: by p=0 score descending, then by p=0.25 score descending
results_df = results_df.sort_values(
    [f"p={P_VALUES[0]}", f"p={P_VALUES[-1]}"],
    ascending=[False, False]
)

# Save main table
results_path = DATA_DIR / "relaxed_modularity_scores.csv"
results_df.to_csv(results_path)
print(f"Saved: {results_path}")

# Save detailed per-layer results
detail_df = pd.DataFrame(detail_records)
detail_path = DATA_DIR / "relaxed_modularity_detail.csv"
detail_df.to_csv(detail_path, index=False)
print(f"Saved: {detail_path}")

print(f"\nResults table ({len(results_df)} objects x {len(P_VALUES)} p-values):")
display(results_df)

In [ ]:
# Cell 11 – Summary statistics and delta analysis
print("=== SUMMARY ===\n")

for p_col, p_val in enumerate(P_VALUES):
    col = f"p={p_val}"
    n_above_zero = (results_df[col] > 0).sum()
    n_ast_above = ((results_df[col] > 0) & (results_df["type"] == "ast")).sum()
    n_blt_above = ((results_df[col] > 0) & (results_df["type"] == "builtin")).sum()
    max_score = results_df[col].max()
    top_scorer = results_df[col].idxmax()
    
    print(f"p = {p_val}:")
    print(f"  Circuits with score > 0: {n_above_zero} / {N_OBJECTS}")
    print(f"    AST:     {n_ast_above}")
    print(f"    Builtin: {n_blt_above}")
    print(f"  Max score: {max_score}/8 ({top_scorer})")
    print()

# Score changes from p=0 to p=0.25
col_strict = f"p={P_VALUES[0]}"
col_relaxed = f"p={P_VALUES[-1]}"
results_df["delta"] = results_df[col_relaxed] - results_df[col_strict]

increased = results_df[results_df["delta"] > 0]
decreased = results_df[results_df["delta"] < 0]
stable = results_df[results_df["delta"] == 0]

print(f"Score change (p=0 -> p=0.25):")
print(f"  Increased: {len(increased)} circuits")
print(f"  Decreased: {len(decreased)} circuits")
print(f"  Stable:    {len(stable)} circuits")

if len(decreased) > 0:
    print(f"\n  Decreased circuits (possible token-driven):")
    for idx, row in decreased.sort_values("delta").iterrows():
        print(f"    {idx}: {row[col_strict]}/8 -> {row[col_relaxed]}/8 (delta={row['delta']})")

if len(increased) > 0:
    print(f"\n  Increased circuits (hidden modularity):")
    for idx, row in increased.sort_values("delta", ascending=False).iterrows():
        print(f"    {idx}: {row[col_strict]}/8 -> {row[col_relaxed]}/8 (delta={row['delta']})")

In [ ]:
# Cell 12 – Validation checks

print("=== VALIDATION ===")
print()

# 6.1 Reproduce original modularity at p=0
print("Check 1: p=0 should match original modularity scores (notebook 4)")
print("  Expected top: Import (7/8), Break (6/8), Pass (6/8), ImportFrom (5/8), Continue (4/8)")
col_strict = f"p={P_VALUES[0]}"
top5 = results_df.nlargest(5, col_strict)[["type", col_strict]]
print(f"  Observed top 5:")
for idx, row in top5.iterrows():
    print(f"    {idx}: {row[col_strict]}/8")
print()

# 6.2 Monotonicity check
print("Check 2: Monotonicity (scores should generally not decrease with increasing p)")
non_monotonic = 0
for obj_idx, obj_name in enumerate(all_objects):
    scores = [results[obj_idx, p_col] for p_col in range(len(P_VALUES))]
    for i in range(1, len(scores)):
        if scores[i] < scores[i-1]:
            print(f"  Non-monotonic: {obj_name} scores={scores}")
            non_monotonic += 1
            break
if non_monotonic == 0:
    print("  All circuits are monotonic.")
else:
    print(f"  {non_monotonic} non-monotonic circuits (may occur when trimming changes ranking order).")
print()

# 6.3 Sanity: p=0.25 should not make everything modular
col_relaxed = f"p={P_VALUES[-1]}"
n_modular_relaxed = (results_df[col_relaxed] > 0).sum()
print(f"Check 3: Circuits with score > 0 at p=0.25: {n_modular_relaxed} / {N_OBJECTS}")
if n_modular_relaxed > 40:
    print("  WARNING: More than 40 circuits are modular at p=0.25. Test may be too lenient.")
else:
    print("  OK: Test is not overly lenient.")
print()

# 6.4 At p=0, bottom 5% = ~5 objects per layer
print(f"Check 4: At p=0, bottom {SIGNIFICANCE*100:.0f}% = ~{int(N_OBJECTS * SIGNIFICANCE)} objects per layer")
print(f"  With 8 layers, an object consistently in bottom 5% scores 8/8.")
print(f"  By chance (independent layers), P(>=7/8) = {8 * 0.05**7 * 0.95 + 0.05**8:.8f}")

In [ ]:
# Cell 13 – Display final table sorted by p=0 score
print(f"Final results: {N_OBJECTS} circuits x {len(P_VALUES)} relaxation parameters")
print(f"Max possible score: {N_LAYERS}/8 at each p\n")
display(results_df.drop(columns=["delta"], errors="ignore"))